# Probability of Default — Walkthrough

This notebook walks through the full PD modeling workflow on the synthetic consumer-loan dataset. It mirrors the production-style script `pd_model.py`, but unpacks each step with explanation, intermediate inspection, and inline visualization.

**Outline**
1. Load and inspect the data
2. Time-based development / out-of-time split
3. Feature engineering and preprocessing pipeline
4. Logistic regression — interpretable benchmark
5. XGBoost — non-linear challenger
6. Validation: discrimination, calibration, lift, stability
7. Discussion of findings and what would change in production

## 1. Setup

Imports, paths, and a quick look at the dataset.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

# Locate the project root and import shared utilities
ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT / '03-shared-utilities'))

from model_evaluation import (
    calibration_table, decile_lift_table,
    evaluate_binary_classifier, population_stability_index,
)
from plotting import (
    plot_roc_curve, plot_calibration, plot_decile_lift,
    plot_feature_importance,
)

DATA_PATH = ROOT / 'data' / 'credit_loans.csv'
print('Loading from', DATA_PATH)

## 2. Load data

The synthetic dataset contains 50,000 consumer loans originated over a two-year window. Each row carries borrower demographics, loan terms, credit-bureau attributes at origination, and a `default_flag` indicating whether the loan went 90+ DPD within 12 months.

If `credit_loans.csv` does not exist yet, run `python data/generate_synthetic_data.py` from the project root.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['origination_date'])
print(f'Rows: {len(df):,}')
print(f'Default rate: {df.default_flag.mean():.2%}')
df.head()

In [ ]:
# Default rate over time — the most important sanity check on a credit dataset
by_month = (df.set_index('origination_date')
              .groupby(pd.Grouper(freq='ME'))['default_flag']
              .mean()
              .rename('default_rate'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_month.index, by_month.values, marker='o', color='#1f4e79')
ax.axhline(df.default_flag.mean(), color='grey', ls='--', lw=1,
           label=f'Overall: {df.default_flag.mean():.2%}')
ax.set_title('Monthly default rate (synthetic data)')
ax.set_ylabel('Default rate'); ax.legend()
plt.show()

## 3. Time-based split

Random train/test splits flatter PD models — they let the model see future vintages during training. The honest split is **out-of-time (OOT)**: hold out the most recent six months entirely and validate against them. That is exactly how the model will be used in production, scoring loans the model has never seen vintage-wise.

In [ ]:
cutoff = df.origination_date.max() - pd.DateOffset(months=6)
dev = df[df.origination_date <= cutoff].copy()
oot = df[df.origination_date >  cutoff].copy()

print(f'Dev sample : {len(dev):>7,}  cutoff <= {cutoff.date()}')
print(f'OOT sample : {len(oot):>7,}  cutoff >  {cutoff.date()}')
print(f'Dev default rate : {dev.default_flag.mean():.2%}')
print(f'OOT default rate : {oot.default_flag.mean():.2%}')

## 4. Feature engineering and preprocessing

We standard-scale numeric inputs and one-hot encode the two categoricals. Both go into the same `ColumnTransformer` so the same pipeline preprocesses training, validation, and OOT data identically — preventing the classic bug of fitting a scaler twice with different statistics.

In [ ]:
NUMERIC_FEATURES = [
    'age', 'annual_income', 'employment_years', 'loan_amount',
    'interest_rate', 'term_months', 'dti_ratio',
    'credit_history_years', 'num_open_accounts',
    'num_delinquencies_2y', 'revolving_utilization',
]
CATEGORICAL_FEATURES = ['home_ownership', 'loan_purpose']
TARGET = 'default_flag'

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'),
            CATEGORICAL_FEATURES),
])

X_dev = dev[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_dev = dev[TARGET]
X_oot = oot[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_oot = oot[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, random_state=42, stratify=y_dev,
)
print(f'train: {len(X_train):,}   val: {len(X_val):,}   OOT: {len(X_oot):,}')

## 5. Logistic regression — interpretable benchmark

The regulator's favorite. Coefficient signs are interpretable, calibration is naturally good when the prior is preserved, and the documentation burden is low. We use `class_weight='balanced'` so the loss does not collapse onto the majority class.

In [ ]:
logit = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced')),
])
logit.fit(X_train, y_train)

p_val_lr = logit.predict_proba(X_val)[:, 1]
p_oot_lr = logit.predict_proba(X_oot)[:, 1]
print(f'Logistic — Val AUC : {roc_auc_score(y_val, p_val_lr):.4f}')
print(f'Logistic — OOT AUC : {roc_auc_score(y_oot, p_oot_lr):.4f}')

## 6. XGBoost — non-linear challenger

Captures interactions and non-linearities the logistic cannot. We constrain depth to 4 and apply `min_child_weight=10` and L2 regularization to keep the model from memorizing noise. `scale_pos_weight` handles the class imbalance.

In [ ]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb = Pipeline([
    ('prep', preprocessor),
    ('clf',  XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
        reg_lambda=1.0, scale_pos_weight=pos_weight,
        random_state=42, eval_metric='logloss',
        tree_method='hist', n_jobs=-1,
    )),
])
xgb.fit(X_train, y_train)

p_val_xgb = xgb.predict_proba(X_val)[:, 1]
p_oot_xgb = xgb.predict_proba(X_oot)[:, 1]
print(f'XGBoost — Val AUC : {roc_auc_score(y_val, p_val_xgb):.4f}')
print(f'XGBoost — OOT AUC : {roc_auc_score(y_oot, p_oot_xgb):.4f}')

## 7. Validation suite

Discrimination (Gini), calibration (predicted vs. observed), lift (rank-ordering quality), and PSI (score stability) — the four-corner validation that any credit model needs to pass before deployment.

In [ ]:
from IPython.display import display

print('Calibration on OOT (XGBoost) — predicted vs. actual default rate:')
cal = calibration_table(y_oot, p_oot_xgb, n_bins=10)
display(cal)

In [ ]:
print('Decile lift on OOT (XGBoost):')
lift = decile_lift_table(y_oot, p_oot_xgb)
display(lift)

In [ ]:
p_train_xgb = xgb.predict_proba(X_train)[:, 1]
psi = population_stability_index(p_train_xgb, p_oot_xgb)
verdict = 'stable' if psi < 0.10 else 'moderate drift' if psi < 0.25 else 'major drift'
print(f'Score PSI (train vs. OOT): {psi:.4f}  ->  {verdict}')

## 8. Visualization

Charts are saved to `charts/` for embedding in the README and the model card PDF. Below we render them inline as well.

In [ ]:
CHARTS = Path().resolve() / 'charts'
CHARTS.mkdir(exist_ok=True)

auc = plot_roc_curve(y_oot, p_oot_xgb,
                     'PD Model — ROC Curve (XGBoost, OOT)',
                     str(CHARTS / 'roc_curve.png'))
print(f'OOT AUC: {auc:.4f}   Gini: {2*auc - 1:.4f}')

from PIL import Image as PILImage
for fname in ['roc_curve.png', 'calibration.png', 'decile_lift.png']:
    p = CHARTS / fname
    if p.exists():
        plt.figure(figsize=(7, 5))
        plt.imshow(PILImage.open(p))
        plt.axis('off')
        plt.show()

## 9. Findings and production considerations

**Discrimination.** XGBoost and the logistic regression land within ~1 Gini point of each other on this synthetic dataset. On real consumer-loan data the gap is usually 2-5 Gini points in favor of the gradient-boosted model — enough to be meaningful for cut-off based decisioning but small enough that the regulator-facing logistic remains a defensible production candidate.

**Calibration.** Both models trained with class re-weighting produce probabilities that are systematically too high, because the loss function sees a 50/50 prior rather than the 12% population rate. For decision purposes (rank-ordering and cut-off) this is fine; for IFRS 9 / CECL consumption the scores must be recalibrated using isotonic or sigmoid regression on a held-out sample.

**Stability.** Score PSI between train and OOT is far below 0.10, indicating the score distribution is stable across the time-based split. This is partly an artifact of synthetic data (no real macro shock); on real data, PSI is the first leading indicator of model decay and the trigger for a recalibration cycle.

**What would change in production.**
- Hyperparameter search via cross-validated grid or Bayesian optimization
- Calibration layer (isotonic or sigmoid) on a held-out fold
- SHAP-based reason codes for adverse-action notices (FCRA / ECOA)
- Segmented models by product type, channel, or risk tier
- Macroeconomic overlay for IFRS 9 forward-looking scenarios
- Fairness testing across protected classes (race/sex proxies, age bands)